# 05 - Mini LoRA SFT (Colab T4)

> 配套笔记：[02. 监督微调 SFT](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/notes/03_训练方法/03_02_SFT.md) ｜ doc §5

把 doc §5 的整套流程跑通：扩词表 + reshape → LoRA 配置 → 自定义 collator → 训练 → 合并 → 推理。

**运行环境**：Colab T4 (16GB) | fp16 | eager attention（T4 不支持 FlashAttn2）

**预计耗时**（T4 实测，含下载）：

| 步骤            | 耗时        |
| --------------- | ----------- |
| 安装依赖         | ~30 s       |
| 模型下载 + 准备 | ~30 s       |
| 训练 (3 epochs) | ~1.5 min    |
| 合并 + 推理     | ~30 s       |
| **总计**        | **~3.5 min** |

**对应 doc 章节**：

| Notebook §          | Doc §                                  |
| ------------------- | -------------------------------------- |
| 1. 模型 + 词表扩展  | §5.1 选模型 + 扩词表 + 适配结构        |
| 2. LoRA 配置        | §5.2 LoRA 配置                         |
| 3. 数据 + collator  | §5.3 数据 + 自定义 collator            |
| 4. 训练             | §5.4 训练                              |
| 5. 合并 + 推理      | §5.5 训完合并 + 推理                   |

## 0. 环境

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers peft datasets accelerate

In [ ]:
# 升级 torchao 以解决 PEFT 兼容性问题
!pip install -U torchao

## 1. 模型 + 词表扩展 + 模型结构适配

> 对应 doc §5.1

Qwen2.5-0.5B + T4 fp16 + eager attention，加业务 token 触发 resize，最后开 PEFT + checkpoint 三件套。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# T4 不支持 bf16 必须 fp16; attn_implementation="eager" 是 PyTorch 原生算法 (T4 不支持 FlashAttn2)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    attn_implementation="eager",
    device_map="auto",
)
print(f"模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")
print(f"原始词表大小: {len(tokenizer)}")

In [ ]:
# 扩词表 (业务自定义 token, 触发 resize)
extra_tokens = ["<|product_id|>", "<|order_id|>"]
n_added = tokenizer.add_special_tokens({"additional_special_tokens": extra_tokens})
print(f"加了 {n_added} 个新 token, 词表 V → {len(tokenizer)}")

if n_added > 0:
    # embedding / lm_head [V, d] → [V+k, d], 新行用现有 embedding 均值初始化
    model.resize_token_embeddings(len(tokenizer))
    print(f"resize 后 embedding shape: {model.get_input_embeddings().weight.shape}")

In [ ]:
# 训练前置三件套
model.gradient_checkpointing_enable()    # 省显存: 不存中间激活, backward 时重算
model.enable_input_require_grads()       # PEFT + checkpoint 必须开, 否则 LoRA 收不到梯度
model.config.use_cache = False           # 训练时关 KV cache (和 checkpoint 冲突)

## 2. LoRA 配置

> 对应 doc §5.2

`r=8, lora_alpha=16` (α=2r 工程惯例)；T4 资源紧只加 q_proj + v_proj (LoRA 论文最优配置)。

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 3. 数据 + 自定义 collator

> 对应 doc §5.3

模拟一份小 SFT 数据集 + 手写 **response-only mask collator** (按 §2.4 mask 规则)。

In [ ]:
from datasets import Dataset

# 演示用的小 SFT 数据 (实际从 jsonl 加载)
raw = [
    {"question": "什么是 LLM？", "answer": "LLM 是基于 Transformer 的大语言模型，通过大规模预训练学到了语言知识。"},
    {"question": "1+1=?",        "answer": "1+1=2"},
    {"question": "推荐一本入门 ML 的书", "answer": "可以看《动手学深度学习》, 边学边写代码效果好。"},
    {"question": "什么是 SFT？",   "answer": "SFT 是监督微调，用 (prompt, response) 对训练模型听指令。"},
    {"question": "Python 怎么读 json 文件？", "answer": "用 json.load(open('file.json'))"},
]
# 复制几遍凑训练量
raw = raw * 30
dataset = Dataset.from_list(raw).map(lambda e: {
    "text": (f"<|im_start|>user\n{e['question']}<|im_end|>\n"
             f"<|im_start|>assistant\n{e['answer']}<|im_end|>")
})
print(f"样本数: {len(dataset)}")
print(f"第 0 条:\n{dataset[0]['text']}")

In [ ]:
ASSIST_START = "<|im_start|>assistant\n"
assist_ids = tokenizer.encode(ASSIST_START, add_special_tokens=False)

def collator(batch, max_len=256):
    enc = tokenizer([b["text"] for b in batch],
                    padding=True, truncation=True, max_length=max_len,
                    return_tensors="pt")
    labels = enc["input_ids"].clone()
    for i, ids in enumerate(enc["input_ids"]):
        # 找 "assistant\n" 起点；之前全部 mask 成 -100
        for j in range(len(ids) - len(assist_ids) + 1):
            if ids[j:j+len(assist_ids)].tolist() == assist_ids:
                labels[i, :j+len(assist_ids)] = -100
                break
        labels[i, enc["attention_mask"][i] == 0] = -100   # padding 也 mask
    return {**enc, "labels": labels}

# 验证 mask 是否正确
batch = collator([dataset[0]])
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"labels (-100 = 不算 loss): {batch['labels'][0].tolist()}")
print(f"参与 loss 的部分:")
print(tokenizer.decode([t for t in batch['labels'][0].tolist() if t != -100]))

## 4. 训练

> 对应 doc §5.4

T4 关键配置：**fp16** (不是 bf16) + batch=2 + grad_accum=8 (等效 batch=16) + max_grad_norm=1.0 (fp16 防爆)。

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./out",
    learning_rate=1e-4,                   # demo 放大一点, 几十步看到 loss 降
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    num_train_epochs=3,
    per_device_train_batch_size=2,        # T4 16GB
    gradient_accumulation_steps=8,        # 等效 batch=16
    fp16=True,                            # T4 必须 fp16
    max_grad_norm=1.0,                    # fp16 防爆
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False           # 重要：防止 Trainer 删掉 collator 需要的 text 列
)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=collator,
)
trainer.train()

In [ ]:
# 保存 LoRA adapter
trainer.save_model("./out/adapter")

import os
files = os.listdir("./out/adapter")
total = sum(os.path.getsize(f"./out/adapter/{f}") for f in files if os.path.isfile(f"./out/adapter/{f}"))
print(f"adapter 文件: {files}")
print(f"adapter 总大小: {total/1e6:.2f} MB (对比基座 ~1 GB)")

## 5. 合并 LoRA + 推理 smoke test

> 对应 doc §5.5

`merge_and_unload()` 把 BA 加回 W₀, 之后是普通 HF 模型, 推理零额外开销。

In [ ]:
from peft import PeftModel

# 释放训练时的内存
del model, trainer
torch.cuda.empty_cache()

# 重新加载基座 + 合并 adapter
base = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto"
)
base.resize_token_embeddings(len(tokenizer))    # 基座也要 resize 到 V+k, 否则 shape mismatch
model = PeftModel.from_pretrained(base, "./out/adapter")
model = model.merge_and_unload()                # W' = W₀ + (α/r)·BA
print("合并完成, 现在是普通 HF 模型")

In [ ]:
# 推理 smoke test
model.eval()
model.config.use_cache = True   # 推理时改回 True

prompt = "<|im_start|>user\n什么是 SFT？<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Prompt: 什么是 SFT？")
print(f"模型生成: {generated}")

## 离真实 SFT 的差距

上面这版只为跑通流程，离真实业务 SFT 差不少：

| 维度       | 这版 demo               | 真实业务 SFT                        |
| ---------- | ----------------------- | ----------------------------------- |
| 模型规模   | 0.5B                    | 7B ~ 70B+                           |
| 数据量     | ~150 条 (5 条 × 30)     | 1M ~ 10M 条                         |
| 训练步数   | ~30 steps               | 几万 ~ 几十万                       |
| LoRA 配置  | r=8, q+v                | r=16~32, all-linear                 |
| Collator   | 简化版 (单 ChatML 模板) | 业务自己写, 多角色 mask             |
| Eval       | 没有                    | held-out + benchmark (MMLU 等)      |

本质上和上面的 demo 是同一套流程，规模放大、加 eval 即可。